In [0]:
from pyspark.sql.types import StringType
from pyspark.sql import Window

from pyspark.sql.functions import trim, col, row_number, when, to_date, length

In [0]:
%python

df = spark.read.table('workspace.bronze.crm_sales_details')



In [0]:

for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))



In [0]:

dt_columns = [col_name for col_name in df.columns if col_name.endswith('dt')]

for column_name in dt_columns:
   df = df.withColumn(
      column_name,
      when(
         length(col(column_name)) != 8,
         None
      ).otherwise(
         to_date(col(column_name).cast("string"), "yyyyMMdd")
      )
   )  



In [0]:
RENAME_MAP = {
    "sls_order_num":"order_number",
    "sls_prd_key":"product_key",
    "sls_cust_id":"customer_id",
    "sls_order_dt":"order_date",
    "sls_ship_dt":"ship_date",
    "sls_due_dt":"due_date",
    "sls_sales":"sales",
    "sls_quantity":"quantity",
    "sls_price":"price"
}

In [0]:
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)


In [0]:
df = (
    df
    .withColumn(
        "price",
        when(
            (col("price").isNull()) | (col("price") <= 0),
            when(
                col("quantity") != 0,
                col("sales") / col("quantity")
            ).otherwise(None)
        ).otherwise(col("price"))
    )
)


In [0]:
df.limit(10).display()

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.crm_sales")